In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Build Constructors Dimension
# MAGIC
# MAGIC 1. Read silver `constructors` table
# MAGIC 1. Read gold `ref_nationality_region` table
# MAGIC 1. Join the data from `constructors` with `ref_nationality_region` using `nationality`
# MAGIC 1. Select the required columns
# MAGIC     - constructors.constructor_id
# MAGIC     - constructors.constructor_name
# MAGIC     - constructors.nationality
# MAGIC     - ref_nationality_region.region
# MAGIC 1. Write the transformed data to gold `dim_constructors` table
# MAGIC

In [0]:
%run ../00-common/01.environment_config

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_constructors"

In [0]:
constructors_df = spark.table(f"{catalog_name}.{silver_schema}.constructors")
nationality_region_df = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")



In [0]:
joined_df = constructors_df \
    .join(
        nationality_region_df,
        constructors_df.nationality == nationality_region_df.nationality,
        "left"
    ) \
    .select(
        constructors_df.constructor_id,
        constructors_df.constructor_name,
        constructors_df.nationality,
        nationality_region_df.region.alias("nationality_region")
    )

In [0]:
( joined_df
.write
.format("delta")
.mode("overwrite")
.saveAsTable(target_table) )

In [0]:
spark.table(target_table).display()